# IPC transport benchmark — plots

Reads the CSVs `bench/run_ipc_isolated.sh` writes to `bench/results/` and
renders the graphs used in the top-level `README.md`: end-to-end latency
percentiles, sustained throughput, and the full per-bucket latency
distribution for both IPC transports (SPSC shared-memory ring buffer vs.
`AF_UNIX SOCK_SEQPACKET`).

Re-run `sudo bash bench/run_ipc_isolated.sh` first to regenerate the CSVs, then
run this notebook top to bottom to regenerate the PNGs in `bench/plots/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BENCH_DIR = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd() / "bench"
RESULTS_DIR = BENCH_DIR / "results"
PLOTS_DIR = BENCH_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Palette: fixed categorical slots (blue = ring buffer, orange = unix socket)
# so color identity stays consistent across every chart in this notebook.
RING_COLOR = "#2a78d6"
UNIX_COLOR = "#eb6834"
SURFACE = "#fcfcfb"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "text.color": TEXT_PRIMARY,
    "axes.labelcolor": TEXT_SECONDARY,
    "axes.edgecolor": BASELINE,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": GRID,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlecolor": TEXT_PRIMARY,
    "axes.titleweight": "bold",
})

In [ ]:
summary = pd.read_csv(RESULTS_DIR / "ipc_summary.csv").set_index("variant")
hist_ring = pd.read_csv(RESULTS_DIR / "ipc_histogram_ring.csv")
hist_unix = pd.read_csv(RESULTS_DIR / "ipc_histogram_unix.csv")
summary

## Latency percentiles + throughput

`max` is left out deliberately — both transports show a scheduler-noise tail
that swamps the steady-state comparison the percentiles below are actually
measuring.

In [ ]:
def fmt_ns(v):
    return f"{v:,.0f} ns"


def fmt_rate(v):
    if v >= 1e6:
        return f"{v / 1e6:.2f}M"
    return f"{v / 1e3:.0f}K"


fig, (ax_lat, ax_thr) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

percentile_cols = ["p50_ns", "p95_ns", "p99_ns", "p99.9_ns"]
labels = ["p50", "p95", "p99", "p99.9"]
x = np.arange(len(labels))
width = 0.36

ring_vals = summary.loc["ring", percentile_cols].to_numpy(dtype=float)
unix_vals = summary.loc["unix", percentile_cols].to_numpy(dtype=float)

bars_ring = ax_lat.bar(x - width / 2, ring_vals, width, color=RING_COLOR, label="Ring buffer (shared memory)")
bars_unix = ax_lat.bar(x + width / 2, unix_vals, width, color=UNIX_COLOR, label="AF_UNIX socket")

for bars, vals in ((bars_ring, ring_vals), (bars_unix, unix_vals)):
    for bar, v in zip(bars, vals):
        ax_lat.text(bar.get_x() + bar.get_width() / 2, v * 1.15, fmt_ns(v),
                     ha="center", va="bottom", fontsize=8.5, color=TEXT_SECONDARY)

ax_lat.set_yscale("log")
ax_lat.set_ylim(top=max(ring_vals.max(), unix_vals.max()) * 3)
ax_lat.set_xticks(x)
ax_lat.set_xticklabels(labels)
ax_lat.set_ylabel("End-to-end latency, ns (log scale)")
ax_lat.set_title("Producer → consumer latency")
ax_lat.grid(axis="y", linewidth=0.6, alpha=0.7)
ax_lat.set_axisbelow(True)
for spine in ("top", "right"):
    ax_lat.spines[spine].set_visible(False)
ax_lat.legend(frameon=False, loc="upper left", fontsize=9)

variants = ["ring", "unix"]
thr_vals = summary.loc[variants, "throughput_msgs_per_sec"].to_numpy(dtype=float)
bars_thr = ax_thr.bar(["Ring buffer", "AF_UNIX"], thr_vals, color=[RING_COLOR, UNIX_COLOR], width=0.55)
for bar, v in zip(bars_thr, thr_vals):
    ax_thr.text(bar.get_x() + bar.get_width() / 2, v * 1.15, f"{fmt_rate(v)} msg/s",
                 ha="center", va="bottom", fontsize=9, color=TEXT_SECONDARY)
ax_thr.set_yscale("log")
ax_thr.set_ylim(top=thr_vals.max() * 4)
ax_thr.set_ylabel("Throughput, msg/s (log scale)")
ax_thr.set_title("Sustained throughput")
ax_thr.grid(axis="y", linewidth=0.6, alpha=0.7)
ax_thr.set_axisbelow(True)
for spine in ("top", "right"):
    ax_thr.spines[spine].set_visible(False)

fig.suptitle(f"200MB ITCH slice, {int(summary.loc['ring', 'count']):,} messages — producer/consumer pinned to separate cores",
             fontsize=9.5, color=MUTED, y=1.02)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ipc_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Full latency distribution

The percentiles above are single points on this shape. The ring buffer's mass
sits almost entirely in the 128–256ns bucket; the socket path has nothing below
1,024ns, with almost all of its mass in 1,024–8,192ns — the syscall-plus-two-copies
cost is a shift of the whole distribution, not just its tail.

In [ ]:
merged = pd.merge(
    hist_ring[["bucket_lower_ns", "bucket_upper_ns", "count"]].rename(columns={"count": "ring"}),
    hist_unix[["bucket_lower_ns", "bucket_upper_ns", "count"]].rename(columns={"count": "unix"}),
    on=["bucket_lower_ns", "bucket_upper_ns"],
    how="outer",
).sort_values("bucket_lower_ns").fillna(0)

# Power-of-two buckets with zero messages in *both* transports are outside
# either distribution's support and would just pad the chart with empty bars.
merged = merged[(merged["ring"] > 0) | (merged["unix"] > 0)].reset_index(drop=True)


def bucket_label(lower, upper):
    return f"{lower:,.0f}–{upper:,.0f}"


labels = [bucket_label(l, u) for l, u in zip(merged["bucket_lower_ns"], merged["bucket_upper_ns"])]
x = np.arange(len(merged))
width = 0.4

fig, ax = plt.subplots(figsize=(11, 5), dpi=150)
ax.bar(x - width / 2, merged["ring"], width, color=RING_COLOR, label="Ring buffer (shared memory)")
ax.bar(x + width / 2, merged["unix"], width, color=UNIX_COLOR, label="AF_UNIX socket")

ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8.5)
ax.set_ylabel("Messages per bucket (log scale)")
ax.set_xlabel("End-to-end latency bucket, ns (power-of-two width)")
ax.set_title("End-to-end latency distribution, per power-of-two bucket")
ax.grid(axis="y", linewidth=0.6, alpha=0.7)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.legend(frameon=False, loc="upper right", fontsize=9)

fig.tight_layout()
fig.savefig(PLOTS_DIR / "ipc_latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()